# 📈 决策树 Decision Tree

> 随机森林是一种包含多个决策树的分类器。随机森林的算法是由Leo Breiman和Adele Cutler发展推论出的。随机森林，顾名思义就是用随机的方式建立一个森林，森林里面由很多的决策树组成，而这些决策树之间没有关联。

> 随机森林就是用过集成学习的思想将多棵树集成的一种算法，它的基本单元是决策树，而它的本质属于机器学习的一大分支------集成学习（Ensemble Learning）方法。集成学习就是使用一系列学习器进行学习，并将各个学习方法通过某种特定的规则进行整合，以获得比单个学习器更好的学习效果。集成学习通过建立几个模型，并将它们组合起来来解决单一预测问题。它的工作原理主要是生成多个分类器或者模型，各自独立地学习和作出预测。

> 随机森林是由多棵决策树构成的。对于每棵树，他们使用的训练集是采用放回的方式从总的训练集中采样出来的。而在训练每棵树的结点时，使用的特征是从所有特征中采用按照一定比例随机地无放回的方式抽取的。


> 1. 决策树                      Decision Tree
> 2. 随机森林（Bagging）         Random Forest
> 3. AdaBoost
> 4. Blending
> 5. Stacking


# 💾 数据集介绍

> 本课程将使用在线股票数据,数据包含以下特征:

## bak_daily 接口数据
> * '`ts_code`':        股票代码
> * '`trade_date`':     交易日期
> * '`name`':           股票名称
> * '`pct_change`':     涨跌幅
> * '`close`':          收盘价
> * '`open`':           开盘价
> * '`high`':           最高价
> * '`low`':            最低价
> * '`pre_close`':      昨收价
> * '`vol_ratio`':      量比
> * '`turn_over`':      换手率
> * '`swing`':          振幅
> * '`vol`':            成交量
> * '`amount`':         成交额
> * '`selling`':        内盘
> * '`buying`':         外盘
> * '`total_share`':    总股本(万)
> * '`float_share`':    流通股本(万)
> * '`pe`':             市盈(动)
> * '`industry`':       所属行业
> * '`area`':           所属地域
> * '`float_mv`':       流通市值
> * '`total_mv`':       总市值
> * '`avg_price`':      平均价
> * '`strength`':       强弱度(%)
> * '`activity`':       活跃度(%)
> * '`avg_turnover`':   笔换手
> * '`attack`':         攻击波(%)
> * '`interval_3`':     近3月涨幅
> * '`interval_6`':     近6月涨幅

# 📤 库导入

In [1]:
# 安装必要的库文件
!pip install mlxtend
!pip install hvplot
!pip install tushare

# 核心工具库
import pandas as pd
import numpy as np

# 画图分析库
import matplotlib.pyplot as plt
import seaborn as sns
import hvplot.pandas

# 数据采集库
import tushare as ts

# 其他设置
%matplotlib inline
sns.set_style("whitegrid")

## 💾 数据获取

In [ ]:
trade_date = '20210927'
next_date = '20210927'
test_date = '20210928'
# 获取数据
pro = ts.pro_api('90ebe1169771a7a44ff42d7afb61e243f257ff653fc1c2d10af45410')
# bak_daily 函数接口
# ts_code	    str	    N	    股票代码
# trade_date	str	    N	    交易日期
# end_date	    str	    N	    结束日期
# offset	    str	    N	    开始行数
# limit         str	    N	    最大行数
df = pro.bak_daily(trade_date=trade_date)
ndf = pro.bak_daily(trade_date=next_date) # 一次性获取全部日k线数据
tdf = pro.bak_daily(trade_date=test_date)

#df.head()

df.sort_values("ts_code",inplace=True)
ndf.sort_values("ts_code",inplace=True)
tdf.sort_values("ts_code",inplace=True)
ii = 0
for index, row in df.iterrows():
    #print(ndf.iloc[index].at['pct_change'])
    try:
        df.iat[index, 3] = ndf.loc[ndf['ts_code'] == df.iat[index, 0]].iat[0, 3]
    except Exception as e:
        pass
    ii=ii+1

for index, row in ndf.iterrows():
    #print(ndf.iloc[index].at['pct_change'])
    try:
        ndf.iat[index, 3] = tdf.loc[tdf['ts_code'] == ndf.iat[index, 0]].iat[0, 3]
    except Exception as e:
        pass
    ii=ii+1
    #print(df.loc[df['ts_code'] == row.ts_code]['pct_change'])
    # df.iloc[index, :] = ndf.loc[ndf['ts_code'] == row.ts_code].pct_change

## 💾 数据保存

In [ ]:
# 创建文件
filename = './sample_data/' + trade_date + '.csv'

# 写入数据
df = df.drop(columns=['interval_3', 'interval_6'])
ndf = ndf.drop(columns=['interval_3', 'interval_6'])
df.to_csv(filename, mode="w", encoding='utf-8')
df.head()

## 💾 检查数据

In [ ]:
ts_data = pd.read_csv(filename)
ts_data["industry"] = pd.factorize(ts_data["industry"])[0].astype(np.uint16)
ts_data["area"] = pd.factorize(ts_data["area"])[0].astype(np.uint16)
#ts_data = ts_data.drop(columns=['ts_code', 'trade_date','name','Unnamed: 0','close','open','turn_over','strength','buying',
#                                'high','low','amount','pre_close','total_mv','avg_turnover','total_share','selling','change'])
ts_data = ts_data.drop(columns=['ts_code', 'trade_date','name','Unnamed: 0','close','open','turn_over','strength','buying', 'area',
                                'high','low','amount','pre_close','total_mv','avg_turnover','total_share','selling','change','industry'])
nts_data = ndf.drop(columns=['ts_code', 'trade_date','name','close','open','turn_over','strength','buying', 'area',
                                'high','low','amount','pre_close','total_mv','avg_turnover','total_share','selling','change','industry'])
print("+++++++++++++++++++++++++数据特征信息+++++++++++++++++++++++++++")
print(ts_data.columns)
print("+++++++++++++++++++++++++数据类型信息+++++++++++++++++++++++++++")
ts_data.info()
print("+++++++++++++++++++++++++数据基本信息+++++++++++++++++++++++++++")
ts_data.describe()

# 📊 探索性数据分析 (Exploratory Data Analysis, EDA)

In [ ]:
ts_data.columns

In [ ]:
# for new data
plt.figure(figsize=(20, 16))
sns.heatmap(ts_data.corr(), annot=True)

# 📈 训练决策树模型

> 现在让我们开始训练决策树模型! 我们首先需要将数据分解为一个X数组(包含要进行训练的特性)和一个带有目标变量的y数组(在本例中是pct_change列)

## 决策树参数说明：
- `criterion`: 特征选择标准 "gini"或者"entropy"
***
- `splitter`: 特征切分点选择标准，决策树是递归地选择最优切分点，spliter是用来指明在哪个集合上来递归，有“best”和“random”两种参数可以选择，best表示在所有特征上递归，适用于数据集较小的时候，random表示随机选择一部分特征进行递归，适用于数据集较大的时候。
***
- `max_depth`: 决策树最大深度
***
- `min_samples_split`: 拆分内部节点所需的最小样本数
***
- `min_samples_leaf`: 叶子节点最小样本数
***
- `min_weight_fraction_leaf`: 在叶节点处的所有输入样本权重总和的最小加权分数，如果不输入则表示所有的叶节点的权重是一致的。
***
- `max_features`: 划分时考虑的最大特征数
***
- `max_leaf_nodes`: 限制最大叶子节点数量，如果为None，则有无限的叶节点
***
- `min_impurity_decrease`: 切分点不纯度最小减少程度，如果某个节点的不纯度减少小于这个值，那么该切分点就会被移除
***
- `min_impurity_split`: 切分点不纯度最小减少程度，如果某个节点的不纯度减少小于这个值，那么该切分点就会被移除
***
- 参考资料：https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier


## X and y arrays

## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split
ts_data = ts_data.dropna(axis=0,how='any')
nts_data = nts_data.dropna(axis=0,how='any')

# X = ts_data.drop(columns=['pct_change'])
# y = ts_data['pct_change']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30)

X_train = ts_data.drop(columns=['pct_change'])
y_train = ts_data['pct_change']
X_test = nts_data.drop(columns=['pct_change'])
y_test = nts_data['pct_change']


threshold = 0

y_train[y_train<threshold] = 0
y_train[y_train>threshold] = 1

y_test[y_test<threshold] = 0
y_test[y_test>threshold] = 1

## 基于信息增益的树模型

- 下载安装包：https://graphviz.gitlab.io/_pages/Download/Download_windows.html
- 环境变量配置：https://jingyan.baidu.com/article/020278115032461bcc9ce598.html
- conda install pydot graphviz

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_clf = DecisionTreeClassifier(criterion="entropy", max_depth=10)
tree_clf.fit(X_train,y_train)

In [ ]:
from sklearn.tree import export_graphviz

export_graphviz(
    tree_clf,
    out_file = "ts_tree.dot",
    feature_names = X_train.columns,
    class_names = ['up','fall'],
    rounded = True,
    filled = True
)

In [ ]:
import pydot

(graph,) = pydot.graph_from_dot_file('ts_tree.dot')
graph.write_png('ts_tree.png')

In [ ]:
from IPython.display import Image
Image(filename='ts_tree.png', width=2000, height=2000)

## 决策树的预测结果的评估
**从测试集创建预测，并创建分类报告和混淆矩阵**

In [ ]:
predictions = tree_clf.predict(X_test)

from sklearn.metrics import classification_report,confusion_matrix

print(classification_report(y_test,predictions))

cm=confusion_matrix(y_test,predictions)
print ("上涨股票预测准确率:",round((cm[1,1])/(cm[1,1]+cm[0,1]),4))
print(cm)

              precision    recall  f1-score   support

         0.0       0.44      0.86      0.58      2049
         1.0       0.46      0.10      0.16      2470

    accuracy                           0.44      4519
   macro avg       0.45      0.48      0.37      4519
weighted avg       0.45      0.44      0.35      4519

上涨股票预测准确率: 0.4572
[[1757  292]
 [2224  246]]


## 基于基尼系数的树模型以及性能评估

In [ ]:
tree_clf = DecisionTreeClassifier(criterion="gini", max_depth=10)
tree_clf.fit(X_train,y_train)

predictions = tree_clf.predict(X_test)

print(classification_report(y_test,predictions))

cm=confusion_matrix(y_test,predictions)
print(cm)
print ("上涨股票预测准确率:",round((cm[1,1])/(cm[1,1]+cm[0,1]),4))

              precision    recall  f1-score   support

         0.0       0.44      0.86      0.58      2049
         1.0       0.41      0.08      0.14      2470

    accuracy                           0.43      4519
   macro avg       0.42      0.47      0.36      4519
weighted avg       0.42      0.43      0.34      4519

[[1755  294]
 [2265  205]]
上涨股票预测准确率: 0.4108


# 随机森林模型

## 随机森林算法参数：
- `n_estimators`: 建立决策树的数量。更多的决策树可以有效增加分类准确性，但运行速度会变慢
***
- `criterion`: 特征选择标准 "gini"或者"entropy"
***
- `max_depth`: 决策树最大深度
***
- `min_samples_split`: 拆分内部节点所需的最小样本数
***
- `min_samples_leaf`: 叶子节点最小样本数
***
- `min_weight_fraction_leaf`: 一个叶节点所需要的(所有输入样本的)权重总和中的最小权重部分。当没有提供sample_weight时，示例具有相同的权重。
***
- `max_features`: 机森林允许单个决策树使用特征的最大数量  Auto/None 简单地选取所有特征 sqrt 总特征数的平方根个 0.2 总特征数20%
***
- `max_leaf_nodes`: 限制最大叶子节点数量，如果为None，则有无限的叶节点
***
- `min_impurity_decrease`: 切分点不纯度最小减少程度，如果某个节点的不纯度减少小于这个值，那么该切分点就会被移除。
***
- `min_impurity_split`: 切分点不纯度最小减少程度，如果某个节点的不纯度减少小于这个值，那么该切分点就会被移除
***
- `bootstrap`: 是否有放回的采样 True或者False
***
- `oob_score`: 是否使用out-of-bag样本估计泛化精度
***
- 参考资料：https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier(n_estimators=6000)
rfc.fit(X_train, y_train)

## 预测和评估

使用模型预测y_test的值并评估我们的模型

In [ ]:
rfc_pred = rfc.predict(X_test)
cr = classification_report(y_test,predictions)     # 现在根据结果创建分类报告
print(cr)
# 显示预测的混淆矩阵
cm = confusion_matrix(y_test,rfc_pred)
print(cm)
print ("上涨股票预测准确率:",round((cm[1,1])/(cm[1,1]+cm[0,1]),4))

## 改变决策树数量，查看混淆矩阵的准确性
**特征选择标准'gini' or 'entropy'**

### 基于基尼系数的随机森林

In [ ]:
nsimu = 100
accuracy=[0]*nsimu
ntree = [0]*nsimu
for i in range(1,nsimu):
    rfc = RandomForestClassifier(n_estimators=i*5,min_samples_split=10,max_depth=None,criterion='gini')
    rfc.fit(X_train, y_train)
    rfc_pred = rfc.predict(X_test)
    cm = confusion_matrix(y_test,rfc_pred)
    accuracy[i] = (cm[0,0]+cm[1,1])/cm.sum()
    ntree[i]=i*5

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(x=ntree[1:nsimu],y=accuracy[1:nsimu],s=60,c='red')
plt.title("Number of trees in the Random Forest vs. prediction accuracy (criterion: 'gini')", fontsize=18)
plt.xlabel("Number of trees", fontsize=15)
plt.ylabel("Prediction accuracy from confusion matrix", fontsize=15)

### 基于信息增量的随机森林

In [ ]:
nsimu = 100
accuracy=[0]*nsimu
ntree = [0]*nsimu
for i in range(1,nsimu):
    rfc = RandomForestClassifier(n_estimators=i*5,min_samples_split=10,max_depth=None,criterion='entropy')
    rfc.fit(X_train, y_train)
    rfc_pred = rfc.predict(X_test)
    cm = confusion_matrix(y_test,rfc_pred)
    accuracy[i] = (cm[0,0]+cm[1,1])/cm.sum()
    ntree[i]=i*5

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(x=ntree[1:nsimu],y=accuracy[1:nsimu],s=60,c='red')
plt.title("Number of trees in the Random Forest vs. prediction accuracy (criterion: 'entropy')", fontsize=18)
plt.xlabel("Number of trees", fontsize=15)
plt.ylabel("Prediction accuracy from confusion matrix", fontsize=15)

## 改变决策树深度

In [ ]:
nsimu = 100
accuracy=[0]*nsimu
ntree = [0]*nsimu
for i in range(1,nsimu):
    rfc = RandomForestClassifier(n_estimators=i*5,min_samples_split=10,max_depth=None,criterion='gini')
    rfc.fit(X_train, y_train)
    rfc_pred = rfc.predict(X_test)
    cm = confusion_matrix(y_test,rfc_pred)
    accuracy[i] = (cm[0,0]+cm[1,1])/cm.sum()
    ntree[i]=i*5

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(x=ntree[1:nsimu],y=accuracy[1:nsimu],s=60,c='red')
plt.title("Number of trees in the Random Forest vs. prediction accuracy (max depth: None)", fontsize=18)
plt.xlabel("Number of trees", fontsize=15)
plt.ylabel("Prediction accuracy from confusion matrix", fontsize=15)

In [ ]:
nsimu = 100
accuracy=[0]*nsimu
ntree = [0]*nsimu
for i in range(1,nsimu):
    rfc = RandomForestClassifier(n_estimators=i*5,min_samples_split=10,max_depth=5,criterion='entropy')
    rfc.fit(X_train, y_train)
    rfc_pred = rfc.predict(X_test)
    cm = confusion_matrix(y_test,rfc_pred)
    accuracy[i] = (cm[0,0]+cm[1,1])/cm.sum()
    ntree[i]=i*5

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(x=ntree[1:nsimu],y=accuracy[1:nsimu],s=60,c='red')
plt.title("Number of trees in the Random Forest vs. prediction accuracy (max depth: 5)", fontsize=18)
plt.xlabel("Number of trees", fontsize=15)
plt.ylabel("Prediction accuracy from confusion matrix", fontsize=15)

## 最小样本分割准则

In [ ]:
nsimu = 100
accuracy=[0]*nsimu
ntree = [0]*nsimu
for i in range(1,nsimu):
    rfc = RandomForestClassifier(n_estimators=i*5,min_samples_split=2,max_depth=None,criterion='gini')
    rfc.fit(X_train, y_train)
    rfc_pred = rfc.predict(X_test)
    cm = confusion_matrix(y_test,rfc_pred)
    accuracy[i] = (cm[0,0]+cm[1,1])/cm.sum()
    ntree[i]=i*5

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(x=ntree[1:nsimu],y=accuracy[1:nsimu],s=60,c='red')
plt.title("Number of trees in the Random Forest vs. prediction accuracy (minimum sample split: 2)", fontsize=18)
plt.xlabel("Number of trees", fontsize=15)
plt.ylabel("Prediction accuracy from confusion matrix", fontsize=15)

In [ ]:
nsimu = 100
accuracy=[0]*nsimu
ntree = [0]*nsimu
for i in range(1,nsimu):
    rfc = RandomForestClassifier(n_estimators=i*5,min_samples_split=20,max_depth=None,criterion='gini')
    rfc.fit(X_train, y_train)
    rfc_pred = rfc.predict(X_test)
    cm = confusion_matrix(y_test,rfc_pred)
    accuracy[i] = (cm[0,0]+cm[1,1])/cm.sum()
    ntree[i]=i*5

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(x=ntree[1:nsimu],y=accuracy[1:nsimu],s=60,c='red')
plt.title("Number of trees in the Random Forest vs. prediction accuracy (minimum sample split: 20)", fontsize=18)
plt.xlabel("Number of trees", fontsize=15)
plt.ylabel("Prediction accuracy from confusion matrix", fontsize=15)

# AdaBoost模型

## 模型参数：
- `base_estimator`: 用于构建Boost的基估计模型，支持抽样样本的加权，以及适当的classes_和n_classes_属性，默认为使用DecisionTreeClassifier模型，其中max_depth=1。
***
- `n_estimators`: Boost终止时的最大base_estimator数量。在完全匹配的情况下，学习过程会提前停止
***
- `learning_rate`: 在每次增强迭代时应用于每个分类器的学习权重。较高的学习率会增加每个分类器的贡献。在learning_rate和n_estimators参数之间存在权衡。
***
- `random_state`: 控制每次迭代时base_estimator上给定的随机种子。
***
- `algorithm`: 如果SAMME。然后使用SAMME.R实助推算法。Base_estimator必须支持类概率的计算。如果‘SAMME’则使用SAMME离散增强算法。SAMME.R算法的收敛速度通常比SAMME算法快，提高迭代次数少，测试误差小
- 参考资料：https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html

In [ ]:
# 使用单一决策树建模
from sklearn.tree import DecisionTreeClassifier
tree = DecisionTreeClassifier(criterion='entropy',random_state=1,max_depth=1)
from sklearn.metrics import accuracy_score
tree = tree.fit(X_train,y_train)
y_train_pred = tree.predict(X_train)
y_test_pred = tree.predict(X_test)
tree_train = accuracy_score(y_train,y_train_pred)
tree_test = accuracy_score(y_test,y_test_pred)
print('Decision tree train/test accuracies %.3f/%.3f' % (tree_train,tree_test))

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
ada = AdaBoostClassifier(base_estimator=tree,n_estimators=500,learning_rate=0.1,random_state=1)
ada = ada.fit(X_train,y_train)
y_train_pred = ada.predict(X_train)
y_test_pred = ada.predict(X_test)
ada_train = accuracy_score(y_train,y_train_pred)
ada_test = accuracy_score(y_test,y_test_pred)
print('Adaboost train/test accuracies %.3f/%.3f' % (ada_train,ada_test))

In [ ]:
# 画出单层决策树与Adaboost的决策边界
cr = classification_report(y_test,y_test_pred)     # 现在根据结果创建分类报告
print(cr)
# 显示预测的混淆矩阵
cm = confusion_matrix(y_test,y_test_pred)
print(cm)
print ("上涨股票预测准确率:",round((cm[1,1])/(cm[1,1]+cm[0,1]),4))

Blending集成学习算法
## Blending算法参数：
- `base_estimator`: 用于构建Boost的基估计模型，支持抽样样本的加权，以及适当的classes_和n_classes_属性，默认为使用DecisionTreeClassifier模型，其中max_depth=1。
***
- `n_estimators`: Boost终止时的最大base_estimator数量。在完全匹配的情况下，学习过程会提前停止
***
- `learning_rate`: 在每次增强迭代时应用于每个分类器的学习权重。较高的学习率会增加每个分类器的贡献。在learning_rate和n_estimators参数之间存在权衡。
***
- `random_state`: 控制每次迭代时base_estimator上给定的随机种子。
***
- `algorithm`: 如果SAMME。然后使用SAMME.R实助推算法。Base_estimator必须支持类概率的计算。如果‘SAMME’则使用SAMME离散增强算法。SAMME.R算法的收敛速度通常比SAMME算法快，提高迭代次数少，测试误差小
- 参考资料：https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html

In [ ]:
## 创建训练集和验证集
X_train1,X_val,y_train1,y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=1)
print("The shape of training X:",X_train1.shape)
print("The shape of training y:",y_train1.shape)
print("The shape of test X:",X_test.shape)
print("The shape of test y:",y_test.shape)
print("The shape of validation X:",X_val.shape)
print("The shape of validation y:",y_val.shape)

In [ ]:
#  设置第一层分类器
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

clfs = [SVC(probability = True),RandomForestClassifier(n_estimators=5, n_jobs=-1, criterion='gini'),KNeighborsClassifier()]

# 设置第二层分类器
from sklearn.linear_model import LinearRegression
lr = LinearRegression()

In [ ]:
# 输出第一层的验证集结果与测试集结果
val_features = np.zeros((X_val.shape[0],len(clfs)))  # 初始化验证集结果
test_features = np.zeros((X_test.shape[0],len(clfs)))  # 初始化测试集结果

for i,clf in enumerate(clfs):
    clf.fit(X_train1,y_train1)
    val_feature = clf.predict_proba(X_val)[:, 1]
    test_feature = clf.predict_proba(X_test)[:,1]
    val_features[:,i] = val_feature
    test_features[:,i] = test_feature

In [ ]:
# 将第一层的验证集的结果输入第二层训练第二层分类器
lr.fit(val_features,y_val)
# 输出预测的结果
from sklearn.model_selection import cross_val_score
cross_val_score(lr,test_features,y_test,cv=5)

# Stacking-scikit learn集成学习算法

## 模型参数：

参考资料：https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB 
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import StackingClassifier

RANDOM_SEED = 42

clf1 = KNeighborsClassifier(n_neighbors=1)
clf2 = RandomForestClassifier(random_state=RANDOM_SEED)
clf3 = GaussianNB()
lr = LogisticRegression()

sclf = StackingClassifier(estimators=[('KNN',clf1), ('RF',clf2), ('NB',clf3)],  # 第一层分类器
                          final_estimator=lr              # 第二层分类器
                          )

print('5-fold cross validation:\n')

for clf, label in zip([clf1, clf2, clf3, sclf], ['KNN', 'Random Forest', 'Naive Bayes','StackingClassifier']):
    scores = cross_val_score(clf, X_train.values, y_train.values, cv=5, scoring='accuracy', error_score="raise")
    print("Accuracy: %0.2f (+/- %0.2f) [%s]" % (scores.mean(), scores.std(), label))

In [ ]:
# 2.使用概率作为元特征
clf1 = KNeighborsClassifier(n_neighbors=1)
clf2 = RandomForestClassifier(random_state=RANDOM_SEED)
clf3 = GaussianNB()
lr = LogisticRegression()

sclf = StackingClassifier(estimators=[('KNN',clf1), ('RF',clf2), ('NB',clf3)],
                          stack_method='predict_proba',  # 
                          final_estimator=lr)

print('5-fold cross validation:\n')

for clf, label in zip([clf1, clf2, clf3, sclf], 
                      ['KNN', 
                       'Random Forest', 
                       'Naive Bayes',
                       'StackingClassifier']):

    scores = cross_val_score(clf, X_train, y_train, 
                                              cv=5, scoring='accuracy')
    print("Accuracy: %0.2f (+/- %0.2f) [%s]" 
          % (scores.mean(), scores.std(), label))

# Stacking-mlxtend 集成学习算法

## 模型参数：
参考资料：http://rasbt.github.io/mlxtend/user_guide/classifier/StackingClassifier/

In [ ]:
# 3. 堆叠5折CV分类与网格搜索(结合网格搜索调参优化)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from mlxtend.classifier import StackingCVClassifier

# Initializing models

clf1 = KNeighborsClassifier(n_neighbors=1)
clf2 = RandomForestClassifier(random_state=1)
clf3 = GaussianNB()
lr = LogisticRegression()

sclf = StackingCVClassifier(classifiers=[clf1, clf2, clf3], 
                            meta_classifier=lr)

params = {'kneighborsclassifier__n_neighbors': [1, 5],
          'randomforestclassifier__n_estimators': [10, 50],
          'meta_classifier__C': [0.1, 10.0]}

grid = GridSearchCV(estimator=sclf, 
                    param_grid=params, 
                    cv=5,
                    refit=True)

grid.fit(X_train.values, y_train.values)

cv_keys = ('mean_test_score', 'std_test_score', 'params')

for r, _ in enumerate(grid.cv_results_['mean_test_score']):
    print("%0.3f +/- %0.2f %r"
          % (grid.cv_results_[cv_keys[0]][r],
             grid.cv_results_[cv_keys[1]][r] / 2.0,
             grid.cv_results_[cv_keys[2]][r]))

print('Best parameters: %s' % grid.best_params_)
print('Accuracy: %.2f' % grid.best_score_)